In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_score, recall_score, f1_score
)
import timm
import cv2
from NoduleDS import NoduleDataset

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 916   # change to 284 or 916 for other runs
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [18]:
# ── Configuration ─────────────────────────────────────────────────────────────
BATCH_SIZE    = 16
NUM_EPOCHS    = 30
LEARNING_RATE = 3e-4
IMAGE_SIZE    = 224
NUM_WORKERS   = 4
WEIGHT_DECAY  = 0.01
ATTENTION_WEIGHT = 1.0
SIGMA_SCALE   = 1.0

MODEL_NAME    = f'spatial_attention_only_seed{SEED}.pth'

# Path to the fine-tuned ResNet-50 baseline checkpoint
BASELINE_CHECKPOINT = './resnet50_binary.pth'

split_base_path  = './dataset_nodule21/cxr_images/proccessed_data/split_data'
train_images_path = f'{split_base_path}/train/images'
val_images_path   = f'{split_base_path}/val/images'
test_images_path  = f'{split_base_path}/test/images'

# No-aug CSV for training (required by spatial attention due to bbox alignment)
train_csv_path = f'{split_base_path}/train/metadata_no_aug.csv'
val_csv_path   = f'{split_base_path}/val/metadata_val.csv'
test_csv_path  = f'{split_base_path}/test/metadata_test.csv'

In [19]:
# ── Transforms ────────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ── Datasets & Loaders ────────────────────────────────────────────────────────
print('Loading datasets...')
train_df = pd.read_csv(train_csv_path)
val_df   = pd.read_csv(val_csv_path)
test_df  = pd.read_csv(test_csv_path)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

# return_bbox=True — spatial attention loss needs bbox coordinates
train_dataset = NoduleDataset(train_df, train_images_path, transform=train_transform, return_bbox=True)
val_dataset   = NoduleDataset(val_df,   val_images_path,   transform=val_transform,   return_bbox=True)
test_dataset  = NoduleDataset(test_df,  test_images_path,  transform=val_transform,   return_bbox=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

Loading datasets...
Train: 3657 | Val: 782 | Test: 785


In [20]:
# ── Spatial Attention Module ───────────────────────────────────────────────────
class SpatialAttentionModule(nn.Module):
    def __init__(self, in_channels=2048, reduction=8):
        super().__init__()
        self.conv1    = nn.Conv2d(in_channels, in_channels // reduction, kernel_size=1)
        self.bn1      = nn.BatchNorm2d(in_channels // reduction)
        self.relu     = nn.ReLU(inplace=True)
        self.conv2    = nn.Conv2d(in_channels // reduction, in_channels // reduction, kernel_size=3, padding=1)
        self.bn2      = nn.BatchNorm2d(in_channels // reduction)
        self.conv_out = nn.Conv2d(in_channels // reduction, 1, kernel_size=1)
        self.sigmoid  = nn.Sigmoid()

    def forward(self, x):
        att = self.relu(self.bn1(self.conv1(x)))
        att = self.relu(self.bn2(self.conv2(att)))
        attention = self.sigmoid(self.conv_out(att))
        return attention, x * attention


# ── Baseline ResNet-50 + Spatial Attention only (no MST blocks) ───────────────
class ResNet50_SpatialAttentionOnly(nn.Module):
    """
    Ablation variant: ResNet-50 backbone (frozen) + Spatial Attention (trainable)
    No MST blocks.
    """
    def __init__(self, num_classes=2, dropout=0.1):
        super().__init__()

        # ── ResNet-50 backbone (will be frozen) ───────────────────────────────
        backbone = timm.create_model('resnet50', pretrained=False, num_classes=0)
        self.stem   = nn.Sequential(backbone.conv1, backbone.bn1, backbone.act1, backbone.maxpool)
        self.stage1 = backbone.layer1
        self.stage2 = backbone.layer2
        self.stage3 = backbone.layer3
        self.stage4 = backbone.layer4

        # ── Spatial Attention (TRAINABLE) ─────────────────────────────────────
        self.spatial_attention = SpatialAttentionModule(in_channels=2048)

        # ── Classifier (TRAINABLE) ────────────────────────────────────────────
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier  = nn.Linear(2048, num_classes)

    def forward(self, x, return_attention=False):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)   # [B, 2048, 7, 7]

        attention_map, x = self.spatial_attention(x)

        x = self.global_pool(x).flatten(1)
        out = self.classifier(x)

        if return_attention:
            return out, attention_map
        return out

    def get_attention_map(self, x):
        with torch.no_grad():
            x = self.stem(x)
            x = self.stage1(x)
            x = self.stage2(x)
            x = self.stage3(x)
            x = self.stage4(x)
            attention_map, _ = self.spatial_attention(x)
        return attention_map

In [21]:
# ── Load model and pretrained weights ─────────────────────────────────────────
print('Initialising model...')
model = ResNet50_SpatialAttentionOnly(num_classes=2, dropout=0.1)

print(f'Loading pretrained ResNet-50 weights from {BASELINE_CHECKPOINT}...')
checkpoint = torch.load(BASELINE_CHECKPOINT, map_location='cpu')
pretrained_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint

# Map old ResNet-50 keys to new model structure
model_dict = model.state_dict()
pretrained_dict_filtered = {}

for k, v in pretrained_dict.items():
    if k.startswith('conv1'):
        pretrained_dict_filtered['stem.0.' + k] = v
    elif k.startswith('bn1'):
        pretrained_dict_filtered['stem.1.' + k] = v
    elif k.startswith('layer1'):
        pretrained_dict_filtered['stage1.' + k[7:]] = v
    elif k.startswith('layer2'):
        pretrained_dict_filtered['stage2.' + k[7:]] = v
    elif k.startswith('layer3'):
        pretrained_dict_filtered['stage3.' + k[7:]] = v
    elif k.startswith('layer4'):
        pretrained_dict_filtered['stage4.' + k[7:]] = v
    elif k == 'fc.weight':
        pretrained_dict_filtered['classifier.weight'] = v
    elif k == 'fc.bias':
        pretrained_dict_filtered['classifier.bias'] = v

model_dict.update(pretrained_dict_filtered)
missing, unexpected = model.load_state_dict(model_dict, strict=False)
print(f'  Missing keys (expected — spatial attention is new): {len(missing)}')
print(f'  Unexpected keys: {len(unexpected)}')

model = model.to(device)

# ── Freeze everything EXCEPT spatial_attention and classifier ─────────────────
for name, param in model.named_parameters():
    if 'spatial_attention' in name or 'classifier' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
total     = trainable + frozen
print(f'\nParameter summary:')
print(f'  Trainable : {trainable:,} ({100*trainable/total:.2f}%)')
print(f'  Frozen    : {frozen:,} ({100*frozen/total:.2f}%)')
print(f'  Total     : {total:,}')
print('\n✓ Frozen    : ResNet-50 backbone')
print('✓ Trainable : Spatial Attention module + Classifier')

Initialising model...
Loading pretrained ResNet-50 weights from ./resnet50_binary.pth...
  Missing keys (expected — spatial attention is new): 0
  Unexpected keys: 6

Parameter summary:
  Trainable : 1,120,003 (4.55%)
  Frozen    : 23,508,032 (95.45%)
  Total     : 24,628,035

✓ Frozen    : ResNet-50 backbone
✓ Trainable : Spatial Attention module + Classifier


In [22]:
# ── Loss, optimiser, scheduler ────────────────────────────────────────────────
train_labels_arr = train_df['label'].values
class_counts  = np.bincount(train_labels_arr)
class_weights = len(train_labels_arr) / (len(class_counts) * class_counts)
class_weights = torch.FloatTensor(class_weights).to(device)
print(f'Class weights — No Nodule: {class_weights[0]:.4f} | Nodule: {class_weights[1]:.4f}')

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer  = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

Class weights — No Nodule: 0.6971 | Nodule: 1.7684


In [23]:
# ── Attention loss helpers (same as resnet-50-guidedv3) ───────────────────────
def bbox_to_gaussian_mask(bbox, target_size=(7, 7), sigma_scale=1.0):
    batch_size = bbox.shape[0]
    h, w = target_size
    y_coords = torch.linspace(0, 1, h, device=bbox.device).view(-1, 1).expand(h, w)
    x_coords = torch.linspace(0, 1, w, device=bbox.device).view(1, -1).expand(h, w)
    masks = []
    for i in range(batch_size):
        x, y, box_w, box_h = bbox[i]
        cx = x + box_w / 2
        cy = y + box_h / 2
        sigma_x = max((box_w / 2) * sigma_scale, 0.05)
        sigma_y = max((box_h / 2) * sigma_scale, 0.05)
        gaussian = torch.exp(
            -((x_coords - cx) ** 2) / (2 * sigma_x ** 2) -
            ((y_coords - cy) ** 2) / (2 * sigma_y ** 2)
        )
        masks.append(gaussian)
    return torch.stack(masks).unsqueeze(1)


def spatial_attention_loss(attention_map, bbox, label, sigma_scale=1.0):
    positive_mask = label == 1
    if positive_mask.sum() == 0:
        return torch.tensor(0.0, device=attention_map.device)
    pos_attention = attention_map[positive_mask]
    pos_bbox      = bbox[positive_mask]
    target_size   = (pos_attention.shape[2], pos_attention.shape[3])
    gaussian_masks = bbox_to_gaussian_mask(pos_bbox, target_size, sigma_scale)
    mse_loss = F.mse_loss(pos_attention, gaussian_masks)
    att_prob = pos_attention / (pos_attention.sum(dim=(2, 3), keepdim=True) + 1e-8)
    tgt_prob = gaussian_masks / (gaussian_masks.sum(dim=(2, 3), keepdim=True) + 1e-8)
    kl_loss  = F.kl_div((att_prob + 1e-8).log(), tgt_prob, reduction='batchmean')
    return 0.5 * mse_loss + 0.5 * kl_loss

In [24]:
# ── Training and validation functions ─────────────────────────────────────────
def train_epoch(model, loader, criterion, optimizer, device, attention_weight, sigma_scale):
    model.train()
    running_loss = running_cls = running_att = 0.0
    correct = total = 0
    all_preds, all_labels, all_probs = [], [], []

    pbar = tqdm(loader, desc='Train')
    for images, labels, bboxes in pbar:
        images, labels, bboxes = images.to(device), labels.to(device), bboxes.to(device)
        optimizer.zero_grad()

        outputs, attention_map = model(images, return_attention=True)
        cls_loss = criterion(outputs, labels)
        att_loss = spatial_attention_loss(attention_map, bboxes, labels, sigma_scale)
        loss     = cls_loss + attention_weight * att_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        running_cls  += cls_loss.item()
        running_att  += att_loss.item()

        probs = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        total   += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().detach().numpy())

        pbar.set_postfix({
            'loss': f'{running_loss/len(pbar):.4f}',
            'cls' : f'{running_cls/len(pbar):.4f}',
            'att' : f'{running_att/len(pbar):.4f}',
            'acc' : f'{100.*correct/total:.2f}%'
        })

    return (
        running_loss / len(loader),
        running_cls  / len(loader),
        running_att  / len(loader),
        100. * correct / total,
        all_preds, all_labels, all_probs
    )


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = total = 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels, bboxes in tqdm(loader, desc='Val/Test'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            running_loss += loss.item()
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    epoch_loss = running_loss / len(loader)
    epoch_acc  = 100. * correct / total
    precision  = precision_score(all_labels, all_preds, average='binary', zero_division=0)
    recall     = recall_score(all_labels, all_preds, average='binary', zero_division=0)
    f1         = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    auc        = roc_auc_score(all_labels, all_probs)

    return epoch_loss, epoch_acc, all_preds, all_labels, all_probs, precision, recall, f1, auc

In [25]:
# ── Training Loop ─────────────────────────────────────────────────────────────
print('\n' + '='*70)
print('Starting training — Spatial Attention Only (seed={SEED})')
print('='*70)

best_val_f1 = 0.0

for epoch in range(NUM_EPOCHS):
    print(f'\nEpoch {epoch+1}/{NUM_EPOCHS}')
    print('-' * 70)

    train_loss, train_cls, train_att, train_acc, train_preds, train_labels, train_probs = train_epoch(
        model, train_loader, criterion, optimizer, device, ATTENTION_WEIGHT, SIGMA_SCALE
    )
    train_auc = roc_auc_score(train_labels, train_probs)

    val_loss, val_acc, val_preds, val_labels, val_probs, val_prec, val_recall, val_f1, val_auc = validate(
        model, val_loader, criterion, device
    )

    scheduler.step()

    print(f'Train — Loss: {train_loss:.4f} (Cls: {train_cls:.4f}, Att: {train_att:.4f}) | Acc: {train_acc:.2f}% | AUC: {train_auc:.4f}')
    print(f'Val   — Loss: {val_loss:.4f} | Acc: {val_acc:.2f}% | Prec: {val_prec:.4f} | Recall: {val_recall:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}')
    print(f'LR: {optimizer.param_groups[0]["lr"]:.6f}')

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            'epoch'              : epoch,
            'model_state_dict'   : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc'            : val_acc,
            'val_auc'            : val_auc,
            'val_precision'      : val_prec,
            'val_recall'         : val_recall,
            'val_f1'             : val_f1,
            'seed'               : SEED,
        }, MODEL_NAME)
        print(f'✓ Saved best model (Val F1: {val_f1:.4f} | Val Recall: {val_recall:.4f})')

print('\nTraining complete!')


Starting training — Spatial Attention Only (seed={SEED})

Epoch 1/30
----------------------------------------------------------------------


Val/Test: 100%|██████████| 49/49 [00:15<00:00,  3.14it/s]


Train — Loss: 2.0909 (Cls: 0.7093, Att: 1.3817) | Acc: 62.65% | AUC: 0.6708
Val   — Loss: 0.6117 | Acc: 67.39% | Prec: 0.4631 | Recall: 0.8356 | F1: 0.5959 | AUC: 0.7959
LR: 0.000299
✓ Saved best model (Val F1: 0.5959 | Val Recall: 0.8356)

Epoch 2/30
----------------------------------------------------------------------


Val/Test: 100%|██████████| 49/49 [00:13<00:00,  3.54it/s]


Train — Loss: 1.8196 (Cls: 0.6297, Att: 1.1899) | Acc: 65.16% | AUC: 0.7072
Val   — Loss: 0.5705 | Acc: 73.53% | Prec: 0.5273 | Recall: 0.7733 | F1: 0.6270 | AUC: 0.8282
LR: 0.000297
✓ Saved best model (Val F1: 0.6270 | Val Recall: 0.7733)

Epoch 3/30
----------------------------------------------------------------------


...

Train — Loss: 1.3456 (Cls: 0.5649, Att: 0.7807) | Acc: 67.95% | AUC: 0.7818
Val   — Loss: 0.5263 | Acc: 74.81% | Prec: 0.5414 | Recall: 0.8133 | F1: 0.6501 | AUC: 0.8526
LR: 0.000003

Epoch 29/30
----------------------------------------------------------------------


Val/Test: 100%|██████████| 49/49 [00:11<00:00,  4.17it/s]


Train — Loss: 1.3209 (Cls: 0.5600, Att: 0.7609) | Acc: 68.83% | AUC: 0.7892
Val   — Loss: 0.5306 | Acc: 74.81% | Prec: 0.5402 | Recall: 0.8356 | F1: 0.6562 | AUC: 0.8531
LR: 0.000001

Epoch 30/30
----------------------------------------------------------------------


Val/Test: 100%|██████████| 49/49 [00:11<00:00,  4.13it/s]

Train — Loss: 1.3171 (Cls: 0.5640, Att: 0.7531) | Acc: 68.23% | AUC: 0.7817
Val   — Loss: 0.5312 | Acc: 74.42% | Prec: 0.5364 | Recall: 0.8178 | F1: 0.6479 | AUC: 0.8442
LR: 0.000000

Training complete!


In [26]:
# ── Load best model and evaluate on test set ──────────────────────────────────
print('\n' + '='*70)
print('Loading best model for test evaluation...')
print('='*70)

checkpoint = torch.load(MODEL_NAME, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1}")
print(f"Val F1: {checkpoint['val_f1']:.4f} | Val Recall: {checkpoint['val_recall']:.4f}")

test_loss, test_acc, test_preds, test_labels, test_probs, test_prec, test_recall, test_f1, test_auc = validate(
    model, test_loader, criterion, device
)

print('\n' + '='*70)
print(f'TEST SET RESULTS — Spatial Attention Only (seed={SEED})')
print('='*70)
print(f'Test Loss      : {test_loss:.4f}')
print(f'Test Accuracy  : {test_acc:.2f}%')
print(f'Test AUC-ROC   : {test_auc:.4f}')
print(f'Test Precision : {test_prec:.4f}')
print(f'Test Recall    : {test_recall:.4f}')
print(f'Test F1-Score  : {test_f1:.4f}')

print('\nClassification Report:')
print(classification_report(test_labels, test_preds, target_names=['No Nodule', 'Nodule'], digits=4))

print('Confusion Matrix:')
print(confusion_matrix(test_labels, test_preds))


Loading best model for test evaluation...
Loaded best model from epoch 5
Val F1: 0.6783 | Val Recall: 0.8667


Val/Test: 100%|██████████| 50/50 [00:13<00:00,  3.76it/s]


TEST SET RESULTS — Spatial Attention Only (seed=916)
Test Loss      : 0.5900
Test Accuracy  : 73.38%
Test AUC-ROC   : 0.8324
Test Precision : 0.5119
Test Recall    : 0.7926
Test F1-Score  : 0.6221

Classification Report:
              precision    recall  f1-score   support

   No Nodule     0.8998    0.7113    0.7945       568
      Nodule     0.5119    0.7926    0.6221       217

    accuracy                         0.7338       785
   macro avg     0.7058    0.7519    0.7083       785
weighted avg     0.7926    0.7338    0.7468       785

Confusion Matrix:
[[404 164]
 [ 45 172]]


In [27]:
# ── Compute 4 attention-bbox alignment metrics ─────────────────────────────────
import SimpleITK as sitk
from PIL import Image
TARGET_SIZE = 224
# ── Resolve image paths for subset ───────────────────────────────────────────
subset_csv_path = './dataset_nodule21/cxr_images/proccessed_data/subset_metadata2.csv'

candidate_image_dirs = [
    './dataset_nodule21/cxr_images/proccessed_data/split_data/test/images',
    './dataset_nodule21/cxr_images/proccessed_data/split_data/val/images',
    './dataset_nodule21/cxr_images/proccessed_data/split_data/train/images',
]

def resolve_img_path(img_name):
    for d in candidate_image_dirs:
        p = os.path.join(d, img_name)
        if os.path.exists(p):
            return p
    return None

subset_df = pd.read_csv(subset_csv_path).copy()
subset_df['resolved_path'] = subset_df['img_name'].apply(resolve_img_path)
missing = subset_df['resolved_path'].isna().sum()
if missing > 0:
    print(f'Warning: dropping {missing} rows with missing image files')
subset_df = subset_df[subset_df['resolved_path'].notna()].reset_index(drop=True)
print(f'Usable rows   : {len(subset_df)}')
print(f'Unique images : {subset_df["img_name"].nunique()}')
print(f'Positive rows : {(subset_df["label"]==1).sum()}')
print('\n' + '='*70)
print('Computing Attention-BBox Alignment Metrics on RNNet-MST (no spatial attention)')
print('='*70)

unique_images = subset_df.drop_duplicates(subset='img_name').reset_index(drop=True)

bbox_coverage_list   = []
detection_rate_list  = []
peak_proximity_list  = []
attention_focus_list = []

for idx in tqdm(range(len(unique_images)), desc='Attention Metrics'):
    row   = unique_images.iloc[idx]
    label = int(row['label'])

    if label == 0:
        continue  # only evaluate on positive (nodule) cases

    # ── Load image ────────────────────────────────────────────────────────────
    image_itk = sitk.ReadImage(row['resolved_path'])
    arr = sitk.GetArrayFromImage(image_itk)
    if len(arr.shape) == 3:
        arr = arr[0]
    arr = arr.astype(np.float32)
    mn, mx = arr.min(), arr.max()
    if mx > mn:
        arr = ((arr - mn) / (mx - mn) * 255).astype(np.uint8)
    else:
        arr = np.zeros_like(arr, dtype=np.uint8)
    arr = np.stack([arr, arr, arr], axis=-1)
    img_tensor = val_transform(Image.fromarray(arr)).unsqueeze(0).to(device)

    # ── Get attention map ─────────────────────────────────────────────────────
    attention_map = model.get_attention_map(img_tensor)          # [1, 1, 7, 7]
    attention_map = attention_map.squeeze().cpu().numpy()         # [7, 7]
    attention_map = cv2.resize(attention_map, (TARGET_SIZE, TARGET_SIZE))  # [224, 224]

    # Normalize to [0, 1]
    a_min, a_max = attention_map.min(), attention_map.max()
    if a_max > a_min:
        attention_map = (attention_map - a_min) / (a_max - a_min)

    # ── Get all bboxes for this image ─────────────────────────────────────────
    img_name = row['img_name']
    img_rows = subset_df[subset_df['img_name'] == img_name]
    img_w    = img_rows.iloc[0].get('img_width',  1024)
    img_h    = img_rows.iloc[0].get('img_height', 1024)

    img_bbox_coverage   = []
    img_detection_rate  = []
    img_peak_proximity  = []
    img_attention_focus = []

    for _, brow in img_rows.iterrows():
        if brow['label'] != 1:
            continue

        x = (brow['x']     / img_w) * TARGET_SIZE
        y = (brow['y']     / img_h) * TARGET_SIZE
        w = (brow['width'] / img_w) * TARGET_SIZE
        h = (brow['height']/ img_h) * TARGET_SIZE

        x_pix = max(0, int(x))
        y_pix = max(0, int(y))
        x_end = min(TARGET_SIZE, int(x + w))
        y_end = min(TARGET_SIZE, int(y + h))

        bbox_mask = np.zeros((TARGET_SIZE, TARGET_SIZE), dtype=np.float32)
        bbox_mask[y_pix:y_end, x_pix:x_end] = 1.0
        bbox_area = bbox_mask.sum()
        if bbox_area == 0:
            continue

        # 1. BBox Coverage
        coverage = (attention_map * bbox_mask).sum() / bbox_area
        img_bbox_coverage.append(float(coverage))

        # 2. Detection Rate
        high_att      = (attention_map > 0.5).astype(np.float32)
        overlap_ratio = (high_att * bbox_mask).sum() / bbox_area
        img_detection_rate.append(1.0 if overlap_ratio > 0.2 else 0.0)

        # 3. Peak Proximity
        peak_y, peak_x = np.unravel_index(np.argmax(attention_map), attention_map.shape)
        bbox_cx  = x + w / 2
        bbox_cy  = y + h / 2
        dist     = np.sqrt((peak_x - bbox_cx)**2 + (peak_y - bbox_cy)**2)
        max_dist = np.sqrt(TARGET_SIZE**2 + TARGET_SIZE**2)
        proximity = max(0.0, 1.0 - dist / max_dist)
        img_peak_proximity.append(float(proximity))

        # 4. Attention Focus
        outside_mask = 1.0 - bbox_mask
        mean_inside  = (attention_map * bbox_mask).sum()  / (bbox_area + 1e-8)
        mean_outside = (attention_map * outside_mask).sum() / (outside_mask.sum() + 1e-8)
        focus = min(mean_inside / (mean_outside + 1e-8), 10.0)
        img_attention_focus.append(float(focus))

    if img_bbox_coverage:
        bbox_coverage_list.append(np.mean(img_bbox_coverage))
        detection_rate_list.append(np.mean(img_detection_rate))
        peak_proximity_list.append(np.max(img_peak_proximity))
        attention_focus_list.append(np.mean(img_attention_focus))

print('\n' + '='*70)
print('ATTENTION-BBOX ALIGNMENT METRICS — RNNet-MST (no spatial attention)')
print('='*70)
print(f'  Images evaluated : {len(bbox_coverage_list)}')
print(f'  BBox Coverage    : {np.mean(bbox_coverage_list):.4f}')
print(f'  Detection Rate   : {np.mean(detection_rate_list):.4f}  ({np.mean(detection_rate_list)*100:.2f}%)')
print(f'  Peak Proximity   : {np.mean(peak_proximity_list):.4f}')
print(f'  Attention Focus  : {np.mean(attention_focus_list):.4f}')

Usable rows   : 171
Unique images : 135
Positive rows : 171

Computing Attention-BBox Alignment Metrics on RNNet-MST (no spatial attention)


Attention Metrics: 100%|██████████| 135/135 [00:04<00:00, 32.84it/s]


ATTENTION-BBOX ALIGNMENT METRICS — RNNet-MST (no spatial attention)
  Images evaluated : 135
  BBox Coverage    : 0.4249
  Detection Rate   : 0.4531  (45.31%)
  Peak Proximity   : 0.7796
  Attention Focus  : 3.0240
